In [ ]:
import stim

import sinter

from stimbposd import BPOSD, sinter_decoders

import numpy as np

import matplotlib.pyplot as plt

import multiprocessing

from pathlib import Path

In [ ]:
import sys
from pathlib import Path

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

from circuit_library import logical_t as _circuit_library

_circuit_library.configure(5)

from circuit_library.logical_t import (
    stabilizers_k_z,
    stabilizers_k_x,
    V_even,
    V_odd,
    S_gate_track,
    stabilizers_s_x,
    stabilizers_s_z,
    stabilizers_added_X,
    stabilizers_k_z_enlargable,
    stabilizers_k_z_enlargable_horizontal,
    stabilizers_k_z_enlargable_vertical,
    stabilizers_k_z_enlarged,
    stabilizers_s_z_enlarged,
    stabilizers_added_Z,
    stabilizer_global,
    stabilizers_k_x_enlargable,
    stabilizers_k_x_enlarged,
    stabilizers_s_x_enlargable,
    stabilizers_s_x_enlarged,
    stabilizers_general,
    stabilizers_k_z_unchange,
    stabilizers_merged,
    stabilizers_merged_without_add_X,
    stabilizers_k_x_unchange,
    stabilizers_s_x_unchange,
    stabilizers_merged_string,
    stabilizers_merged_string_without_add_Z,
    DISTANCE,
    QUBIT_RANGE_3D,
    QUBIT_RANGE_2D,
    QUBIT_RANGE_ALL,
    DEFAULT_SYNDROME_ANCILLA,
    DEFAULT_FLAG_QUBITS,
    FACE_ANCILLAS,
    STRING_ANCILLAS,
    LOGICAL_3D_PAULI,
    LOGICAL_2D_PAULI,
    MEASUREMENT_XX_PRODUCTS,
    LOGICAL_CZ_PAIRS,
    FINAL_2D_Y_PAULI,
    WEIGHT_3_SEQUENCE,
    WEIGHT_4_SEQUENCE,
    WEIGHT_5_SEQUENCE,
    WEIGHT_6_SEQUENCE,
    WEIGHT_7_SEQUENCE,
    X_WEIGHT_8_SEQUENCE,
    X_WEIGHT_9_SEQUENCE,
    X_WEIGHT_12_SEQUENCE,
    X_WEIGHT_13_SEQUENCE,
    X_WEIGHT_18_SEQUENCE,
    X_WEIGHT_19_SEQUENCE,
    MPP_CIRCUITS_BY_TYPE_AND_WEIGHT,
    pauli_targets,
    pauli_string_type,
    pauli_weight,
    parse_flag_data_sequence,
    run_flag_data_sequence,
    MPP_circuit,
    count_added_flag_measurements,
    flag_adjusted_target_rec,
    count_measurements_for_stabilizers,
    target_rec_from_previous_stabilizer_round,
    Stabilizers_measurement_general,
    Stabilizers_measurement_merged,
    Stabilizers_measurement_merged_string,
    detector_k_x,
    Stabilizers_measurement_general_after_gate,
    lattice_surgery_Merge,
    lattice_surgery_Split,
    lattice_surgery_Merge_string,
    lattice_surgery_Split_string,
    lattice_surgery_Reset_string,
    measure_logical_qubits_3D,
    measure_logical_qubits_2D,
    measurement_XX,
    measurement_ZZ,
    S_Z_Gate,
    S_Z_DAG_Gate,
    logical_CZ,
    MPP_circuit_X_weight_3,
    MPP_circuit_X_weight_4,
    MPP_circuit_X_weight_5,
    MPP_circuit_X_weight_6,
    MPP_circuit_X_weight_7,
    MPP_circuit_X_weight_8,
    MPP_circuit_X_weight_9,
    MPP_circuit_X_weight_12,
    MPP_circuit_X_weight_13,
    MPP_circuit_X_weight_18,
    MPP_circuit_X_weight_19,
    MPP_circuit_X_weight_24,
    MPP_circuit_X_weight_25,
    MPP_circuit_Z_weight_3,
    MPP_circuit_Z_weight_4,
    MPP_circuit_Z_weight_5,
    MPP_circuit_Z_weight_6,
    MPP_circuit_Z_weight_7,
    MPP_circuit_Z_weight_8,
    MPP_circuit_Z_weight_9,
    MPP_circuit_Z_weight_12,
    MPP_circuit_Z_weight_13,
    MPP_circuit_Z_weight_18,
    MPP_circuit_Z_weight_19,
    MPP_circuit_Z_weight_24,
    MPP_circuit_Z_weight_25,
)



In [ ]:
def circuit_generate(rate_mea, rate_idle):
    qubit_range_3D = list(range(0, 65))
    qubit_range_2D = list(range(89, 108))
    qubit_range_data = qubit_range_3D + qubit_range_2D
    qubit_range_all = list(range(0, 108))
    observable_list_xx = []
    observable_list_zz = []

    c = stim.Circuit()
    c.append("H", qubit_range_3D)
    c += Stabilizers_measurement_general(0, 1)
    c.append("TICK")

    c.append("H", qubit_range_2D)
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general_after_gate(rate_mea, 1, After_H_2D=True)
    c.append("TICK")

    c += logical_CZ()
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general_after_gate(rate_mea, 2, After_CZ=True)
    c.append("TICK")

    c += lattice_surgery_Merge_string()
    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    c += Stabilizers_measurement_merged_string(rate_mea, 4, First_merging_round=True)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    c += Stabilizers_measurement_merged_string(rate_mea, 5)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    c += Stabilizers_measurement_merged_string(rate_mea, 6)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    c += Stabilizers_measurement_merged_string(rate_mea, 7)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    offset = c.num_measurements
    c += Stabilizers_measurement_merged_string(rate_mea, 8, offset=offset, Observable_list_ZZ=observable_list_zz)
    c.append("TICK")

    c += lattice_surgery_Split_string(rate_mea)
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 9, After_merging_string=True)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 10)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 11)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 12)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 13)
    c.append("TICK")

    c += lattice_surgery_Reset_string()

    c += S_Z_Gate()
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general_after_gate(rate_mea, 15, After_S_3D=True)

    c.append("H", qubit_range_2D)
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general_after_gate(rate_mea, 16, After_H_2D=True)
    c.append("TICK")
    c += lattice_surgery_Merge()

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    c += Stabilizers_measurement_merged(rate_mea, 17, First_merging_round=True)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    c += Stabilizers_measurement_merged(rate_mea, 18)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    c += Stabilizers_measurement_merged(rate_mea, 19)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    c += Stabilizers_measurement_merged(rate_mea, 20)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle)
    offset = c.num_measurements
    c += Stabilizers_measurement_merged(rate_mea, 21, offset=offset, Observable_list_XX=observable_list_xx)
    c.append("TICK")

    c += lattice_surgery_Split(rate_mea)
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 22, After_merging=True)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 23)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 24)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 25)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 26)
    c.append("TICK")

    c.append("H", qubit_range_2D)
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general_after_gate(rate_mea, 27, After_H_2D=True)
    c.append("TICK")

    c += logical_CZ()
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle)
    c += Stabilizers_measurement_general_after_gate(rate_mea, 28, After_CZ=True)
    c.append("TICK")
    c += Stabilizers_measurement_general(0, 29)
    c.append("TICK")

    c.append("MPP", stim.PauliString(FINAL_2D_Y_PAULI), tag="logical_qubits_2D")
    observable_targets_xx = [
        stim.target_rec(index - c.num_measurements)
        for index in observable_list_xx
    ]
    observable_targets_zz = [
        stim.target_rec(index - c.num_measurements)
        for index in observable_list_zz
    ]
    c.append(
        "OBSERVABLE_INCLUDE",
        [stim.target_rec(-1), *observable_targets_xx, *observable_targets_zz],
        0,
    )
    return c



In [ ]:
c = circuit_generate(0.01, 0.01)
# c.diagram("timeline-svg")

In [ ]:
dem = c.detector_error_model()

In [ ]:
len(c.shortest_graphlike_error())

In [ ]:
def generate_tasks():
    for error_rate in [0.00001, 0.00005, 0.0001, 0.00015, 0.0002]:
        yield sinter.Task(
            circuit=circuit_generate(error_rate, error_rate),
            json_metadata={"p": error_rate},
        )



In [ ]:
samples = sinter.collect(
    num_workers=multiprocessing.cpu_count() - 1,
    max_shots=10_000,
    max_errors=100,
    tasks=generate_tasks(),
    decoders=["hypergraph_union_find"],
    custom_decoders=sinter_decoders(),
    save_resume_filepath="circuit_T_d5.csv"
)

In [ ]:
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(  
    ax=ax,  
    stats=samples,  
    group_func=lambda stat: stat.decoder,  # No 'd' available  
    x_func=lambda stat: stat.json_metadata['p']  # Placeholder x since 'p' is unavailable  
)
ax.loglog()
ax.grid()
ax.set_title("Logical Error Rate vs Physical Error Rate")
ax.set_ylabel("Logical Error Probability (per shot)")
ax.set_xlabel("Physical Error Rate")
ax.legend()
plt.show()